In [1]:
import pickle
import os, sys
import numpy as np
# import shutil
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import torch

from network.SmilesImage2Smiles_Network_svae import *
from data_utils import *
from data_utils_smiles import *
from input_output import *

sys.path.append('../../../')
from data_utils_local.data_utils_smiles import *
from data_utils_local.Generalised_data_utils import *

%load_ext autoreload
%autoreload 2

### **Parameters**

In [2]:
path_init = '/home/rkmvu/Dataset/selfies/zinc/'
gpu_device_id = 0# GPU number for multiple GPUs (pytorch takes default 0 or which is availabe next)
device = torch.device("cuda:"+str(gpu_device_id) if torch.cuda.is_available() else "cpu")
# print(torch.cuda.is_available())
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('='*70)
print('Model will use: {}' .format(device))
print('='*70)


# dataset parameters
# num_total_samples = 3963360#3970176
num_train_samples = 3963360#12941195#10500000#100000#10#
num_train_test_samples = 100000#1000#100000#'all'#10#
train_ratio = 0.5
val_ratio = 0.2
test_ratio = 1 - train_ratio - val_ratio
# dataset_name_prefix = 'zinc6m_all4'#'zinc6m_all4_mona'
smiles_char_filename = 'character_set_zinc6m_all4_special_Br_Cl_train_0_5_val_0_2_test_0_3_nspltokens_35_maxstrlen_250_3963360.json'
smiles_max_length_filename = 'smiles_length_zinc6m_all4_special_Br_Cl_train_0_5_val_0_2_test_0_3_nspltokens_35_maxstrlen_250_3963360.txt'
dataset_name = 'canon_smiles_selfies_train_0_5_val_0_2_test_0_3.parquet'
padding = 'right'

train_data_name = ''.join(['train_', str(train_ratio), '_']).replace('.', '_')
test_data_name = ''.join(['test_', str(test_ratio), '_']).replace('.', '_')
val_data_name = ''.join(['val_', str(val_ratio), '_']).replace('.', '_')

token_path_init = '/home/rkmvu/Codes/ICDCIT_submission/vae_smiles/hsvae/cache/max_len_and_tokens'
smiles_char_filepath = ''.join([token_path_init, '/', smiles_char_filename])
smiles_max_length_filepath = ''.join([token_path_init, '/', smiles_max_length_filename])
#+++++++++++++++++++++++++++++++++++++++++++++++++++

# read smiles,  vocabulary and smiles max-length
smiles = PARSE_SMILES([])

padding = padding
# load smiles char vocabulary
smiles_char, smiles_char_map = smiles.load_smiles_char(smiles_char_filepath)
# load smiles max-length
max_smiles_length = smiles.load_smiles_max_length(smiles_max_length_filepath)

Model will use: cuda:0


### **Creating and Loading model**

In [3]:
## VAE_GRU parameters
# dims_input_data = (max_smiles_length, len(smiles_char_map['domain'].keys())+1) 
# dims_output_data = (max_smiles_length, len(smiles_char_map['domain'].keys())+1)
dims_input_data = (max_smiles_length, len(smiles_char_map['domain'].keys())) 
dims_output_data = (max_smiles_length, len(smiles_char_map['domain'].keys()))
num_kernels = [9, 9, 10]#[[11, 13, 15]#11, 13, 15]#[15, 17, 19]#[11, 13, 15]#[9, 9, 10]
size_kernels = [9, 9, 11]#[[13, 11, 9]#9, 9, 11]#[15, 13, 11]#[9, 9, 11]#[9, 9, 11]
num_fc_layer_encoder = 0#2#
dropout_prob = 0.0
dims_latent = 50#25#
act_func = 'relu'
scale_latent_space = 1e-2
num_fc_layer_decoder = 0
gru_hidden_size = 500
num_gru = 4#5#3#
layer_type = '1dcnn_gru'
weight_init = 'xvr_unifrm'
loss_type = 'bce_kld'
solver_type = 'adam'
num_epoch = 500
batch_size = 128
learning_rate = 1e-4#1e-3#
save_result_ateach_epoch = 50
result_savepath = ''.join(['cache/smi2smi_svae/', path_init.split('/')[-2], '_', dataset_name, '_', str(num_train_samples), '/']).replace('.','_')
#+++++++++++++++++++++++++++++++++++++++++++++++++++

smiles_svae_model = SmIm2Sm_Network(dims_input_data, dims_output_data, 
                                   smiles_char_map=smiles_char_map, 
                                   max_string_len=max_smiles_length, 
                                   padding=padding, 
                                   num_kernels=num_kernels, 
                                   size_kernels=size_kernels, 
                                   num_fc_layer_encoder=num_fc_layer_encoder, 
                                   dropout_prob=dropout_prob, 
                                   dims_latent=dims_latent, 
                                   act_func=act_func, 
                                   scale_latent_space=scale_latent_space, 
                                   num_fc_layer_decoder=num_fc_layer_decoder, 
                                   gru_hidden_size=gru_hidden_size, 
                                   num_gru=num_gru, 
                                   layer_type=layer_type, 
                                   device=device, 
                                   weight_init=weight_init, 
                                   loss_type=loss_type, 
                                   solver_type=solver_type, 
                                   num_epoch=num_epoch, 
                                   batch_size=batch_size, 
                                   learning_rate=learning_rate, 
                                   save_result_ateach_epoch=save_result_ateach_epoch, 
                                   result_savepath=result_savepath)
#+++++++++++++++++++++++++++++++++++++++++++++++++++


# test on best trained network
smiles_svae_model.load_test_network()# load pre-trained best network


----------------------------------------------------------------------
Network summary
----------------------------------------------------------------------
Weights "Conv1d(250, 9, kernel_size=(9,), stride=(1,))" initialized by "xavior_uniform" scheme
Weights "Conv1d(9, 9, kernel_size=(9,), stride=(1,))" initialized by "xavior_uniform" scheme
Weights "Conv1d(9, 10, kernel_size=(11,), stride=(1,))" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=90, out_features=50, bias=True)" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=50, out_features=50, bias=True)" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=50, out_features=1, bias=True)" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=50, out_features=50, bias=True)" initialized by "xavior_uniform" scheme
Weights "GRU(50, 500, num_layers=4, batch_first=True)" initialized by "orthogonal" scheme
Weights "Linear(in_features=500, out_features=35, bias=True)" initiali

## **Test the model for reconstruction**

In [4]:
def encode(smiles_or_selfies):
    z =  smiles_svae_model.smiles2latent_vector(smiles=smiles_or_selfies)
    return z

def decode(x_latent):
    x = smiles_svae_model.latent_vector2smiles(x_latent=x_latent)
    return x

def name2smiles(name):
    row = df_temp[df_temp['name'] == name].iloc[0]
    return row['smiles']

def smiles2name(smiles):
    smiles = canonical_fn(smiles)
    row = df_temp[df_temp['smiles'] == smiles].iloc[0]
    return row['name']

def smi2nneigh(smiles, indices, n_neigh=10):
    smiles = canonical_fn(smiles)
    idx = smiles_main.index(smiles)
    neigh_idx = indices[idx][1:n_neigh+1]
    nn_smiles = [smiles_main[i] for i in neigh_idx]
    nn_names = [smiles2name(x) for x in nn_smiles]
    return {'smiles':nn_smiles, 'name':nn_names}


In [5]:
import pandas as pd

smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)
df = pd.read_csv('/home/rkmvu/Dataset/selfies/zinc/properties_dtratio_0.001_test_0_3_canon_smiles_selfies_with_descriptors_train_0_5_val_0_2_test_0_3.csv')
mask = df['SMILES'].apply(smiles_clean_fn)
smiles_main = df[mask]['SMILES'].tolist()
df

,SMILES,MolWt,TPSA,EState_VSA1,NHOHCount,MolLogP,fr_COO,nAcid,ATSC1c,ATSC1se,...,fr_ether,fr_halogen,fr_ketone,fr_nitro,fr_nitro_arom,fr_nitroso,fr_phenol,fr_sulfone,AUTOCORR2D_156,nBase
0,COc1ccc(CN2CCC3CN(C(=O)C(C)C(C)(C)C)C3C2)nn1,346.475,58.56,0.000000,0,2.20010,0,0,-0.330215,0.267548,...,1,0,0,0,0,0,0,0,1.405,1
1,C#CCOC(C)C(=O)N1CC2(CCCN2C(=O)c2ccc(CO)o2)C1,346.383,83.22,6.103966,1,0.62720,0,0,-0.612666,-0.332817,...,1,0,0,0,0,0,0,0,1.405,0
2,CNC(=O)c1ccc(C)c(NC(=O)NC(C)c2ccccc2OCc2ccccc2)c1,417.509,79.46,0.000000,3,4.81632,0,0,-0.750331,-0.201391,...,1,0,0,0,0,0,0,0,1.476,0
3,COC1(C)CCN(C(=O)c2ccccc2SC(F)(F)F)CC1,333.375,29.54,5.508331,0,3.93960,0,0,-0.420343,-0.167045,...,1,3,0,0,0,0,0,0,1.182,0
4,CS(=O)(=O)CCC1NC(=O)N(CCOc2cccc(Cl)c2)C1=O,360.819,92.78,27.817388,1,1.07390,0,0,-0.617955,0.009875,...,1,1,0,0,0,0,0,1,1.245,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7759,CC(C)NCCC(=O)N1CCCCC1c1cc(NC(=O)c2cn3cc(Cl)ccc...,457.966,107.42,0.000000,3,3.40480,0,0,-0.588136,-0.127373,...,0,1,0,0,0,0,0,0,0.835,1
7760,N#Cc1ccc(N2CCCC2C(=O)OCC(=O)Nc2ccc([N+](=O)[O-...,462.384,125.57,52.563041,1,3.63598,0,0,-0.679651,0.251615,...,1,3,0,1,1,0,0,0,1.187,0
7761,CCC(N)CNC(=O)NC(C)CCc1ccc2c(c1)OCO2,307.394,85.61,0.000000,4,1.77290,0,0,-0.794791,-0.296837,...,2,0,0,0,0,0,0,0,1.252,1
7762,COc1cc(C(=O)N2CCC3CCN(Cc4cnsn4)C3C2)sn1,365.484,71.45,0.000000,0,1.73980,0,0,-0.352193,0.126301,...,1,0,0,0,0,0,0,0,0.880,1


### **Reconstruction Accuracy on Test data**

In [6]:
z = encode(smiles_or_selfies=smiles_main)
smiles_recon = decode(x_latent=z)
print(z.shape)

exact = [x==y for x, y in zip(smiles_main, smiles_recon)]
smis, checks = smiles.check_smiles_validity(smiles=smiles_recon)

print('='*80)
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(checks)/z.shape[0]}')
print('='*80)


<===smiles reconstruction===>


  4%|▎         | 2/54 [00:00<00:02, 17.96it/s]

100%|██████████| 54/54 [00:04<00:00, 13.32it/s]


<===smiles reconstruction===>


100%|██████████| 54/54 [00:01<00:00, 27.58it/s]


(6813, 50)
<===checking smiles validity===>


100%|██████████| 6813/6813 [00:02<00:00, 2919.38it/s]

Exact reconstruction: 0.1485395567297813
Valid reconstruction: 0.5504183179216204


## **Test the model for representation**

In [7]:
import selfies as sf
import pandas as pd

smiles2selfie_fn = lambda x: sf.encoder(x)
canonical_fn = lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=False)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)
len_filter_fn = lambda x: sf.len_selfies(x)<=128


#### **Preprocess drugs**

In [8]:
df = pd.read_csv('/home/rkmvu/Dataset/selfies/drug/mdrug.csv')
_, valid_mask = smiles.check_smiles_validity(df['smiles'].tolist())
df = df[valid_mask].copy()
df['smiles'] = df['smiles'].apply(canonical_fn)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

len_mask = df['selfies'].apply(len_filter_fn)
df = df[len_mask]

df_temp = df.copy()
mask = df_temp['smiles'].apply(smiles_clean_fn)
df_temp = df_temp[mask]
df_temp = df_temp.drop_duplicates(subset=['smiles'])
smiles_main = df_temp['smiles'].tolist()
print(f'Number of smiles: {len(smiles_main)}')
df_temp

<===checking smiles validity===>


 71%|███████▏  | 987/1381 [00:00<00:00, 4954.98it/s]

100%|██████████| 1381/1381 [00:00<00:00, 4834.89it/s]


Number of smiles: 1089


,name,smiles,InChl,type,selfies
0,Abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,InChI=1S/C14H18N6O/c15-14-18-12(17-9-2-3-9)11-...,Drug,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...
1,Abiraterone,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,InChI=1S/C26H33NO2/c1-17(28)29-20-10-12-25(2)1...,Drug,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...
2,Acamprosate,CC(=O)NCCCS(=O)(=O)O,"InChI=1S/C5H11NO4S/c1-5(7)6-3-2-4-11(8,9)10/h2...",Drug,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...
3,Acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,InChI=1S/C25H43NO18/c1-6-11(26-8-2-7(3-27)12(3...,Drug,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...
4,Acebutolol,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,InChI=1S/C18H28N2O4/c1-5-6-18(23)20-14-7-8-17(...,Drug,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...
...,...,...,...,...,...
1374,Ziprasidone,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,InChI=1S/C21H21ClN4OS/c22-17-13-18-15(12-20(27...,Drug,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...
1375,Zoledronate,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,"InChI=1S/C5H10N2O7P2/c8-5(15(9,10)11,16(12,13)...",Drug,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...
1377,Zolpidem,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,InChI=1S/C19H21N3O/c1-13-5-8-15(9-6-13)19-16(1...,Drug,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...
1378,Zonisamide,NS(=O)(=O)Cc1noc2ccccc12,"InChI=1S/C8H8N2O3S/c9-14(11,12)5-7-6-3-1-2-4-8...",Drug,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...


In [9]:
# df[df['name'].apply(lambda x: 'tate' in x)]
df_temp['smiles'].nunique()
# len(set(smiles_main))

1089

In [10]:
len(set(df['name'].tolist()))

1348

### **Finding Nearest Neighbours**

In [11]:
from sklearn.neighbors import NearestNeighbors

z = encode(smiles_or_selfies=smiles_main)
n_neigh = NearestNeighbors(n_neighbors=60, metric='euclidean')
n_neigh.fit(z)

<===smiles reconstruction===>


 44%|████▍     | 4/9 [00:00<00:00, 16.21it/s]

100%|██████████| 9/9 [00:00<00:00, 13.47it/s]


NearestNeighbors(metric='euclidean', n_neighbors=60)

In [12]:
distances, indices = n_neigh.kneighbors(z)
distances
indices

array([[   0,  811,   52, ...,  873,  497,  613],
       [   1,  944,  199, ...,  928,  155,  630],
       [   2,   12,  604, ...,  391,  950,  657],
       ...,
       [1086, 1032,  428, ...,   77,  827,  304],
       [1087, 1057,  331, ...,  805,  907,  604],
       [1088,  787,  818, ...,  770, 1022,  133]])

In [13]:
df = pd.DataFrame(z, columns=[f'dim_{i}' for i in range(z.shape[1])])
df.insert(0, 'smiles', smiles_main)
df.insert(0, 'selfies', 'None')
df.insert(0, 'name', 'None')
df['name'] = df['smiles'].apply(smiles2name)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

df2 = pd.read_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_properties.csv')
df3 = pd.merge(df, df2, on='smiles')
# df3.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_props_with_embeds_hvsmi.csv', index=False)
df3.head(2)

,name,selfies,smiles,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,...,VSA_EState3,NHOHCount,NumHDonors,NumHAcceptor,NumRotatableBonds,MolLogP,ATSC1pe,ATSC1are,AATSC1dv,AATSC1are
0,Abacavir,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,0.101602,0.026436,0.000400,-0.066269,0.125349,-0.061688,-0.033620,...,12.615719,4,3,7,4,1.0923,-0.478300,-0.657070,1.039823,-0.015645
1,Abiraterone,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,-0.002068,-0.142546,0.107922,0.268292,-0.042850,-0.148124,0.013788,...,0.000000,0,0,3,2,5.9694,0.296168,0.241524,1.157286,0.003659


In [14]:
## Calculating the Molecular Properties
# smiles_drugs = df['smiles'].tolist()
# all_prop = []
# for drug in smiles_drugs:
#     all_prop.append(calculate_properties(drug))
# df2 = pd.DataFrame(all_prop)
# df2

In [15]:
# df2.insert(0, 'smiles', smiles_drugs)

In [16]:
# # df2.to_csv('./drug_properties.csv', index=False)
# df2

In [17]:
# name2smiles('Abacavir')
# smiles2name('Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1')
# smi2nneigh(name2smiles('Amlodipine'), indices=indices, n_neigh=10)
# smiles_main.index(name2smiles('Amlodipine'))


In [18]:
# smiles = canonical_fn(smiles)
# idx = smiles_main.index(smiles)
# neigh_idx = indices[idx][1:n_neigh+1]
# nn_smiles = [smiles_main[i] for i in neigh_idx]
# nn_names = [smiles2name(x) for x in nn_smiles]

### **Reconstruction Accuracy on Marketed Drugs**

In [19]:
z = encode(smiles_or_selfies=smiles_main)
smiles_recon = decode(x_latent=z)
print(z.shape)

exact = [x==y for x, y in zip(smiles_main, smiles_recon)]
smis, checks = smiles.check_smiles_validity(smiles=smiles_recon)

print('='*80)
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(checks)/z.shape[0]}')
print('='*80)


<===smiles reconstruction===>


 33%|███▎      | 3/9 [00:00<00:00, 19.07it/s]

100%|██████████| 9/9 [00:00<00:00, 15.94it/s]


<===smiles reconstruction===>


100%|██████████| 9/9 [00:00<00:00, 28.11it/s]


(1089, 50)
<===checking smiles validity===>


100%|██████████| 1089/1089 [00:00<00:00, 3015.28it/s]

Exact reconstruction: 0.2194674012855831
Valid reconstruction: 0.45821854912764004


In [20]:
# top_drugs = df_temp['name']
# top_drugs

## **Nearest Neighbour Analysis**

In [21]:
import numpy as np

top_drugs = ["Metformin", "Amoxicillin", "Atorvastatin", "Amlodipine", "Acetaminophen", "Imatinib", "Clozapine", 
             "Ibuprofen", "Azithromycin", "Doxycycline"]
top_drugs = df_temp['name']
n_neigh=50

all_out = {'drug_hvsmi':[], 'nn_idx_hvsmi':[], 'smiles_hvsmi':[], 'names_hvsmi':[]}
for drug in top_drugs:
    _smiles = name2smiles(drug)
    out = smi2nneigh(_smiles, indices=indices, n_neigh=n_neigh)
    drug_name = [drug]*n_neigh
    idx = np.arange(1, n_neigh+1)
    out = {'drug':drug_name, 'nn_idx':idx, **out}
    all_out['drug_hvsmi'].extend(out['drug'])
    all_out['nn_idx_hvsmi'].extend(out['nn_idx'])
    all_out['smiles_hvsmi'].extend(out['smiles'])
    all_out['names_hvsmi'].extend(out['name'])

In [22]:
[len(v) for v in all_out.values()]

[54450, 54450, 54450, 54450]

In [26]:
all_out_df = pd.DataFrame(all_out)
# all_out_df.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/near_smiles_hvsmi.csv', index=False)
all_out_df


,drug_hvsmi,nn_idx_hvsmi,smiles_hvsmi,names_hvsmi
0,Abacavir,1,CCc1ccc(CCOc2ccc(CC3SC(=O)NC3=O)cc2)nc1,Pioglitazone
1,Abacavir,2,CC(C)c1ccc2oc3nc(N)c(C(=O)O)cc3c(=O)c2c1,Amlexanox
2,Abacavir,3,Nc1nc(N)c2nc(-c3ccccc3)c(N)nc2n1,Triamterene
3,Abacavir,4,Nc1nc(CC(=O)Nc2ccc(CCNCC(O)c3ccccc3)cc2)cs1,Mirabegron
4,Abacavir,5,Nc1nc(Cl)nc2c1ncn2C1OC(CO)C(O)C1F,Clofarabine
...,...,...,...,...
54445,Zuclopenthixol,46,Cc1cnc(NC(=O)C2=C(O)c3ccccc3S(=O)(=O)N2C)s1,Meloxicam
54446,Zuclopenthixol,47,C#CC1(O)CCC2C3CCc4cc(OC)ccc4C3CCC21C,Mestranol
54447,Zuclopenthixol,48,CN1CCC2=C(C1)C(c1ccccc1)c1ccccc12,Phenindamine
54448,Zuclopenthixol,49,C#CC1(O)CCC2C3CCc4cc(O)ccc4C3CCC21C,Ethinyl Estradiol


In [24]:
all_out_df.groupby(['drug_hvsmi']).get_group('Amlodipine')#['smiles_hvsmi'].nunique()

,drug_hvsmi,nn_idx_hvsmi,smiles_hvsmi,names_hvsmi
2650,Amlodipine,1,CCNC(=O)CCCC=CCC1C(O)CC(O)C1C=CC(O)CCc1ccccc1,Bimatoprost
2651,Amlodipine,2,CCOC(=O)C1=C(C)NC(C)=C(C(=O)OC)C1c1cccc(Cl)c1Cl,Felodipine
2652,Amlodipine,3,CC(C)OC(=O)CCCC=CCC1C(O)CC(O)C1CCC(O)CCc1ccccc1,Latanoprost
2653,Amlodipine,4,NC(C(=O)NC1C(=O)N2C(C(=O)O)=C(Cl)CCC12)c1ccccc1,Loracarbef
2654,Amlodipine,5,NC(C(=O)NC1C(=O)N2C(C(=O)O)=C(Cl)CSC12)c1ccccc1,Cefaclor
2655,Amlodipine,6,CCC(OC(C)=O)C(CC(C)N(C)C)(c1ccccc1)c1ccccc1,Levomethadyl Acetate
2656,Amlodipine,7,CC(C)OC(=O)CCCC=CCC1C(O)CC(O)C1C=CC(F)(F)COc1c...,Tafluprost
2657,Amlodipine,8,CCCC(=O)OCOC(=O)C1=C(C)NC(C)=C(C(=O)OC)C1c1ccc...,Clevidipine
2658,Amlodipine,9,OC(c1cc(C(F)(F)F)nc2c(C(F)(F)F)cccc12)C1CCCCN1,Mefloquine
2659,Amlodipine,10,NC(=O)CCCNC(=C1C=C(F)C=CC1=O)c1ccc(Cl)cc1,Progabide
